## 1. Hugging Face Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [1]:

# =========================================================
# Phase 1: Hybrid Prefix-LM Legal Model Pretraining
# =========================================================

# --- Cell 1: Install Dependencies ---
!pip uninstall -y transformers huggingface-hub tokenizers safetensors
!pip install "transformers==4.49.0" "huggingface-hub>=0.23.0" safetensors
!pip install ijson

Found existing installation: transformers 4.49.0
Uninstalling transformers-4.49.0:
  Successfully uninstalled transformers-4.49.0
Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2
Found existing installation: tokenizers 0.21.4
Uninstalling tokenizers-0.21.4:
  Successfully uninstalled tokenizers-0.21.4
Found existing installation: safetensors 0.8.0
Uninstalling safetensors-0.8.0:
  Successfully uninstalled safetensors-0.8.0
  Using cached transformers-4.49.0-py3-none-any.whl.metadata (44 kB)
  Using cached huggingface_hub-2.1.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.49.0-py3-none-any.whl (10.

In [2]:

# --- Cell 2: Imports ---
import os
import json
import random
import time
import math
import zipfile
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import ijson
from huggingface_hub import login, hf_hub_download
from tokenizers import ByteLevelBPETokenizer
from transformers import (
    PreTrainedTokenizerFast,
    T5Config,
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer
)
from transformers.trainer_utils import get_last_checkpoint
from datasets import Dataset

## 2. Check GPU

In [3]:
# --- Cell 4: Check GPU ---
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 3. Configuration

#### this is sample configuration, you are free to modify any parameter to train the best performing model

In [ ]:
# --- Cell 3: Configuration & Authentication ---

login(token=os.environ["HF_TOKEN"])

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
OUTPUT_DIR = "./cs781_phase1_hybridencoder"
TOKENIZER_DIR = os.path.join(OUTPUT_DIR, "custom_legal_tokenizer")
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "legal-hybridencoder-baseline")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")

for path in [OUTPUT_DIR, TOKENIZER_DIR, MODEL_OUTPUT_DIR]:
    os.makedirs(path, exist_ok=True)

# Budgets & Architecture (T5-Base Scale ~110M Params)
TRAIN_TOKENS = 200_000_000
VAL_TOKENS = 5_000_000
MAX_SOURCE_LENGTH = 1024
MAX_TARGET_LENGTH = 256

D_MODEL = 768
NUM_LAYERS = 6          # 6 encoder layers, 6 decoder layers
NUM_HEADS = 12
VOCAB_SIZE = 32000

# Decoding Parameters
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.95

In [5]:
# from transformers import AutoTokenizer, T5ForConditionalGeneration
# import torch

# # Replace with your actual Hugging Face model repository ID
# HF_MODEL_REPO = "A-240242/legal-encdec-checkpoints" 

# print("Fetching tokenizer and trained model from Hugging Face Hub...")
# tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_REPO)
# model = T5ForConditionalGeneration.from_pretrained(HF_MODEL_REPO)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
# model.eval()

# print("Model and tokenizer successfully fetched from Hub and ready!")

## 4. Load Tokenizer

In [6]:
# --- Cell 5: Load Tokenizer Corpus & Train Tokenizer ---
sources = {
    "supreme_court": {"file": "Phase_1_Pretraining/Train/train_supreme_court.json", "ratio": 0.45},
    "high_courts": {"file": "Phase_1_Pretraining/Train/train_high_courts.json", "ratio": 0.35},
    "acts": {"file": "Phase_1_Pretraining/Train/train_acts.json", "ratio": 0.15},
    "tribunals": {"file": "Phase_1_Pretraining/Train/train_tribunals.json", "ratio": 0.05}
}

file_paths = {}
for name, config_source in sources.items():
    file_paths[name] = hf_hub_download(repo_id=HF_DATASET_REPO, filename=config_source["file"], repo_type="dataset", token=HF_TOKEN)

def get_tokenizer_corpus(paths, max_docs=1000): 
    for name, path in paths.items():
        print(f"Sampling {max_docs} documents from {name} for tokenizer...")
        count = 0
        with open(path, "rb") as f:
            for record in ijson.items(f, "item"):
                if "text" in record:
                    yield record["text"]
                    count += 1
                if count >= max_docs:
                    break

print("Training custom Byte-Level BPE tokenizer from scratch...")
raw_tokenizer = ByteLevelBPETokenizer()

raw_tokenizer.train_from_iterator(
    get_tokenizer_corpus(file_paths, max_docs=1000),
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=["<|endoftext|>"]
)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=raw_tokenizer._tokenizer,
    bos_token="<|endoftext|>",
    eos_token="<|endoftext|>",
    pad_token="<|endoftext|>"
)
tokenizer.padding_side = "right"  # Standard for T5 Encoder-Decoder
tokenizer.save_pretrained(TOKENIZER_DIR)
print(f"Custom tokenizer saved to {TOKENIZER_DIR}")

Training custom Byte-Level BPE tokenizer from scratch...
Sampling 1000 documents from supreme_court for tokenizer...
Sampling 1000 documents from high_courts for tokenizer...
Sampling 1000 documents from acts for tokenizer...
Sampling 1000 documents from tribunals for tokenizer...



Custom tokenizer saved to ./cs781_phase1_hybridencoder/custom_legal_tokenizer


## 5. Download Training Data From Hugging Face

In [7]:
# --- Cell 6: Stratified Seq2Seq Dataset Generation (OOM-Safe Token Collection) ---
train_sources_data, val_sources_data = [], []
random.seed(42)

for source, config_source in sources.items():
    target_train_tokens = int(TRAIN_TOKENS * config_source["ratio"])
    target_val_tokens = int(VAL_TOKENS * config_source["ratio"])
    
    print(f"Processing Seq2Seq pairs for {source}...")
    source_train_tokens, source_val_tokens = [], []
    
    with open(file_paths[source], "rb") as f:
        for record in ijson.items(f, "item"):
            if "text" not in record:
                continue
            
            tokens = tokenizer.encode(record["text"])
            if len(tokens) < 50:
                continue
                
            if random.random() < 0.05:
                if len(source_val_tokens) < target_val_tokens:
                    source_val_tokens.extend(tokens[:target_val_tokens - len(source_val_tokens)])
            else:
                if len(source_train_tokens) < target_train_tokens:
                    source_train_tokens.extend(tokens[:target_train_tokens - len(source_train_tokens)])
                    
            if len(source_train_tokens) >= target_train_tokens and len(source_val_tokens) >= target_val_tokens:
                break
                
    train_sources_data.extend(source_train_tokens)
    val_sources_data.extend(source_val_tokens)
    print(f"Finished {source}: {len(source_train_tokens)} train tokens, {len(source_val_tokens)} val tokens.")

Processing Seq2Seq pairs for supreme_court...
Finished supreme_court: 90000000 train tokens, 2250000 val tokens.
Processing Seq2Seq pairs for high_courts...
Finished high_courts: 70000000 train tokens, 1750000 val tokens.
Processing Seq2Seq pairs for acts...
Finished acts: 30000000 train tokens, 750000 val tokens.
Processing Seq2Seq pairs for tribunals...
Finished tribunals: 10000000 train tokens, 250000 val tokens.


## 6. Create Train and Validation Tokens Set

## 7. Create Token Blocks

In [8]:
# --- Cell 7 & 8: OOM-Safe Streaming Dataset Creation ---
import gc
from datasets import Dataset

def generate_hybrid_pairs(token_list, max_src, max_tgt):
    chunk_size = max_src + max_tgt
    total_length = (len(token_list) // chunk_size) * chunk_size
    
    for i in range(0, total_length, chunk_size):
        chunk = token_list[i:i + chunk_size]
        src = chunk[:max_src]
        tgt = chunk[max_src:]
        
        # Yield single dictionaries; Dataset.from_generator writes these straight to disk
        yield {
            "input_ids": src + tgt,
            "labels": [-100] * len(src) + tgt,
            "token_type_ids": [0] * len(src) + [1] * len(tgt)
        }

print("Streaming training dataset to disk...")
train_dataset = Dataset.from_generator(
    lambda: generate_hybrid_pairs(train_sources_data, MAX_SOURCE_LENGTH, MAX_TARGET_LENGTH)
)

print("Streaming validation dataset to disk...")
val_dataset = Dataset.from_generator(
    lambda: generate_hybrid_pairs(val_sources_data, MAX_SOURCE_LENGTH, MAX_TARGET_LENGTH)
)

# CRITICAL: Free up the ~7.5 GB of CPU RAM held by the raw token lists
del train_sources_data
del val_sources_data
gc.collect()

print("Datasets successfully streamed to disk!")
print(train_dataset)
print(val_dataset)

Streaming training dataset to disk...


Generating train split: 0 examples [00:00, ? examples/s]

Streaming validation dataset to disk...


Generating train split: 0 examples [00:00, ? examples/s]

Datasets successfully streamed to disk!
Dataset({
    features: ['input_ids', 'labels', 'token_type_ids'],
    num_rows: 156250
})
Dataset({
    features: ['input_ids', 'labels', 'token_type_ids'],
    num_rows: 3906
})


## 8. Create Datasets

In [9]:
# # --- Cell 8: Create Datasets ---
# train_dataset = Dataset.from_dict({
#     "input_ids": train_ids,
#     "labels": train_lbls,
#     "token_type_ids": train_types
# })
# val_dataset = Dataset.from_dict({
#     "input_ids": val_ids,
#     "labels": val_lbls,
#     "token_type_ids": val_types
# })

# print(train_dataset)
# print(val_dataset)

## 9. Define The Model

In [10]:
# --- Cell 9: Define Hybrid Prefix-LM Architecture (8 Layers) ---
import math
import torch
import torch.nn as nn
from transformers import PretrainedConfig
from transformers.modeling_outputs import CausalLMOutput

class HybridPrefixConfig(PretrainedConfig):
    model_type = "hybrid_prefix_lm"
    def __init__(
        self, vocab_size=32000, d_model=768, num_layers=8, 
        num_heads=12, d_ff=2048, pad_token_id=0, eos_token_id=0, **kwargs
    ):
        super().__init__(pad_token_id=pad_token_id, eos_token_id=eos_token_id, **kwargs)
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.d_ff = d_ff

class HybridPrefixLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(config.vocab_size, config.d_model, padding_idx=config.pad_token_id)
        
        # Sized to 2048 to prevent IndexError during generation
        self.pos_embedding = nn.Embedding(2048, config.d_model)
        
        layer = nn.TransformerEncoderLayer(
            d_model=config.d_model, nhead=config.num_heads, 
            dim_feedforward=config.d_ff, dropout=0.1, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=config.num_layers)
        
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight

    def forward(self, input_ids, token_type_ids=None, labels=None, **kwargs):
        batch_size, seq_len = input_ids.size()
        
        # FIX: Extract device dynamically from the scattered tensor
        device = input_ids.device 
        
        mask = torch.triu(torch.full((seq_len, seq_len), float('-inf'), device=device), diagonal=1)
        mask = mask.unsqueeze(0).expand(batch_size, -1, -1).clone()

        if token_type_ids is not None:
            q_is_prefix = (token_type_ids == 0).unsqueeze(2)
            k_is_prefix = (token_type_ids == 0).unsqueeze(1)
            prefix_block = q_is_prefix & k_is_prefix
            mask = mask.masked_fill(prefix_block, 0.0)

        mask = mask.unsqueeze(1).expand(-1, self.config.num_heads, -1, -1)
        mask = mask.reshape(batch_size * self.config.num_heads, seq_len, seq_len)

        positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, -1)
        emb = self.embedding(input_ids) * math.sqrt(self.config.d_model) + self.pos_embedding(positions)

        memory = self.encoder(emb, mask=mask)
        logits = self.lm_head(memory)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

        return CausalLMOutput(loss=loss, logits=logits)

    def generate(self, input_ids, max_new_tokens=384, temperature=0.7, top_p=0.9, repetition_penalty=1.3):
        self.eval()
        batch_size = input_ids.size(0)
        
        # FIX: Extract device dynamically
        device = input_ids.device
        
        curr_ids = input_ids
        curr_types = torch.zeros_like(input_ids)
        unfinished = torch.ones(batch_size, dtype=torch.long, device=device)

        for _ in range(max_new_tokens):
            outputs = self(input_ids=curr_ids, token_type_ids=curr_types)
            next_logits = outputs.logits[:, -1, :]

            # --- 3-Gram Blocking (Prevents Phrase Loops) ---
            no_repeat_ngram_size = 3
            if curr_ids.size(1) >= no_repeat_ngram_size - 1:
                for b in range(batch_size):
                    seq = curr_ids[b].tolist() 
                    tail = seq[-(no_repeat_ngram_size - 1):]
                    
                    for i in range(len(seq) - (no_repeat_ngram_size - 1)):
                        if seq[i : i + (no_repeat_ngram_size - 1)] == tail:
                            banned_token = seq[i + (no_repeat_ngram_size - 1)]
                            next_logits[b, banned_token] = -float("Inf")
            # ----------------------------------------------------

            if repetition_penalty != 1.0:
                score = torch.gather(next_logits, 1, curr_ids)
                score = torch.where(score > 0, score / repetition_penalty, score * repetition_penalty)
                next_logits.scatter_(1, curr_ids, score)

            if temperature > 0:
                next_logits = next_logits / temperature
                if top_p < 1.0:
                    sorted_logits, sorted_indices = torch.sort(next_logits, descending=True)
                    cum_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
                    to_remove = cum_probs > top_p
                    to_remove[..., 1:] = to_remove[..., :-1].clone()
                    to_remove[..., 0] = 0
                    indices_to_remove = to_remove.scatter(1, sorted_indices, to_remove)
                    next_logits[indices_to_remove] = -float("Inf")
                    
                probs = torch.softmax(next_logits, dim=-1)
                next_tokens = torch.multinomial(probs, num_samples=1)
            else:
                next_tokens = torch.argmax(next_logits, dim=-1, keepdim=True)

            curr_ids = torch.cat([curr_ids, next_tokens], dim=-1)
            curr_types = torch.cat([curr_types, torch.ones((batch_size, 1), dtype=torch.long, device=device)], dim=-1)

            unfinished = unfinished & (next_tokens.squeeze(-1) != self.config.eos_token_id).long()
            if unfinished.max() == 0:
                break

        return curr_ids

# Initialize Hybrid Architecture
config = HybridPrefixConfig(vocab_size=len(tokenizer), pad_token_id=tokenizer.pad_token_id, num_layers=NUM_LAYERS)
model = HybridPrefixLM(config)
print(f"Hybrid Prefix-LM Size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

Hybrid Prefix-LM Size: 59.23M parameters


In [11]:
# import torch
# from huggingface_hub import hf_hub_download
# from safetensors.torch import load_file
# from transformers import AutoTokenizer

# HF_MODEL_REPO = "A-240242/legal-hybridencoder-checkpoints"

# print("Fetching tokenizer and trained model weights from Hugging Face Hub...")
# # tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_REPO)

# # 1. Re-initialize your custom model architecture
# model = CustomLegalEncoderDecoder(
#     vocab_size=len(tokenizer),
#     d_model=768,  
#     num_layers=6,  
#     num_heads=12,  
#     pad_token_id=tokenizer.pad_token_id,
#     eos_token_id=tokenizer.eos_token_id,
# )
# 
# # 2. Download and load the saved safetensors weights from your Hub repo
# model_weight_path = hf_hub_download(
#     repo_id=HF_MODEL_REPO, 
#     filename="model.safetensors"
# )
# model.load_state_dict(load_file(model_weight_path))

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
# model.eval()

# print("Custom model and tokenizer successfully fetched from Hub and ready!")

In [12]:
# TOKENIZER_DIR = "./cs781_phase1_encdec/custom_legal_tokenizer"
# print("Loading local tokenizer...")
# tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_DIR)

## 10. Training Arguments

In [13]:
# --- Cell 10: Training Arguments ---
from transformers import default_data_collator

# Keep your existing TrainingArguments definition exactly the same
training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    report_to="none",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    weight_decay=0.1,
    warmup_steps=500,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    push_to_hub=True,
    hub_model_id="A-240242/legal-hybridencoder-checkpoints",
    hub_strategy="checkpoint",
    hub_token=HF_TOKEN,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=100,
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=default_data_collator, 
)

## 11. Training

In [14]:
# --- Cell 11: Training ---
EPOCHS = training_args.num_train_epochs
start_time = time.time()

MOUNTED_CHECKPOINT_DIR = "/kaggle/input/legal-hybridencoder-checkpoints"
latest_checkpoint = (
    get_last_checkpoint(MOUNTED_CHECKPOINT_DIR)
    if os.path.exists(MOUNTED_CHECKPOINT_DIR)
    else None
)

if latest_checkpoint is not None:
  print(f"Auto-detected previous run. Resuming directly from: {latest_checkpoint}")
else:
  print("No prior checkpoint found. Starting training from scratch.")

# Break the memory sharing between embedding and lm_head for your custom model
if hasattr(model, "lm_head") and hasattr(model, "embedding"):
  if model.lm_head.weight.data_ptr() == model.embedding.weight.data_ptr():
    model.lm_head.weight = nn.Parameter(model.lm_head.weight.clone())
    print("Successfully un-tied custom model weights for safetensors.")

trainer.train(resume_from_checkpoint=latest_checkpoint)

training_time = time.time() - start_time
tokens_per_epoch = len(train_dataset) * (MAX_SOURCE_LENGTH + MAX_TARGET_LENGTH)
total_tokens = tokens_per_epoch * EPOCHS
tokens_per_second = total_tokens / training_time

print(f"Training time: {training_time / 3600:.2f} hours")
print(f"Training speed: {tokens_per_second:,.0f} tokens/sec")

No prior checkpoint found. Starting training from scratch.
Successfully un-tied custom model weights for safetensors.


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

## 12. Perplexity

In [ ]:
# --- Cell 12: Perplexity ---
eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)
print(f"Final Validation Loss: {eval_loss:.6f}")
print(f"Final Validation Perplexity: {perplexity:.4f}")

## 13. Plot Training and Validation Loss

In [ ]:
# --- Cell 13: Plot Training and Validation Loss ---
logs = trainer.state.log_history
train_steps, train_loss = [], []
eval_steps, eval_loss_history = [], []

for log in logs:
    if "loss" in log and "step" in log:
        train_steps.append(log["step"])
        train_loss.append(log["loss"])
    if "eval_loss" in log and "step" in log:
        eval_steps.append(log["step"])
        eval_loss_history.append(log["eval_loss"])

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_loss, label="Training Loss")
plt.plot(eval_steps, eval_loss_history, label="Validation Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Encoder-Decoder T5 Training and Validation Loss (200M Tokens)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
print(f"Loss plot saved to: {LOSS_PLOT_PATH}")



## 14. Save Final Model to Kaggle Output

In [ ]:
# --- Cell 14: Save Final Model to Kaggle Output ---
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print(f"Final model saved to {MODEL_OUTPUT_DIR}")

## 15. Push Model to Hugging Face Hub

In [ ]:
# model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)

# print("Model and tokenizer pushed successfully.")

## 16. Load Test Data from HuggingFace

In [ ]:
tokenizer.padding_side = "left"

In [ ]:
# --- Cell 16: Load Test Data and Generate Predictions ---
import json
import torch
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename="Phase_1_Pretraining/Test/test.json", repo_type="dataset", token=HF_TOKEN)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

model.eval()
model.to(device)

predictions = {}
items = list(test_data.items())

for start in tqdm(range(0, len(items), 32), desc="Generating Hybrid Predictions"):
    batch = items[start:start + 32]
    batch_ids = [str(eid) for eid, _ in batch]
    batch_prompts = [record["text"] for _, record in batch]

    # Tokenize the prompt
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SOURCE_LENGTH)
    input_ids = inputs["input_ids"].to(device)

    with torch.inference_mode():
        # Call the custom Hybrid generator
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=384,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3
        )

    # Slice off the prompt to only decode the newly generated text
    prompt_len = input_ids.shape[1]
    generated_sequences = outputs[:, prompt_len:]

    for i, example_id in enumerate(batch_ids):
        generated_text = tokenizer.decode(generated_sequences[i], skip_special_tokens=True)
        generated_words = generated_text.split()[:TARGET_GENERATION_WORDS]
        predictions[example_id] = {"text": " ".join(generated_words)}

## 17. Load Test Prompts

## 18. Load Final Model for Generation

## 19. Generate 128 Words
#### (words, not tokens)

## 20. Generate Predictions for the Test Set

## 21. Save prediction.json

In [ ]:
# --- Cell 21 & 22: Save prediction.json & Create Submission ZIP ---
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print("prediction.json saved successfully:")
print(PREDICTION_PATH)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")
    
print("CodaBench submission created successfully:")
print(SUBMISSION_ZIP)

## 22. Create CodaBench Submission ZIP